# Time-Series Forecasting with ARIMA
**Daily demand • Decomposition • ADF • ARIMA selection • 30-day forecast**

Place `daily-demand-series.csv` in the same directory as this notebook.

## Checklist
- [x] Trend, seasonal and residual decomposition (weekly period=7)
- [x] ADF stationarity test and differencing assessment
- [x] ARIMA hyperparameter grid search
- [x] Chronological holdout evaluation
- [x] MAPE and RMSE
- [x] 30-day future forecast with prediction intervals

In [ ]:
import warnings, itertools
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 1. Load and validate the supplied data

In [ ]:
df = pd.read_csv("daily-demand-series.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")
df = df.asfreq("D")
print(df.info())
display(df.head(), df.tail())
print("Missing values:\\n", df.isna().sum())
y = df["demand"].astype(float)
if y.isna().any():
    y = y.interpolate(method="time")
print("Date range:", y.index.min(), "to", y.index.max(), "| observations:", len(y))

## 2. Chronological train/test split

In [ ]:
# Last 20% (6 observations) held out; never shuffle time-series data.
test_size = max(1, int(round(len(y) * 0.20)))
train, test = y.iloc[:-test_size], y.iloc[-test_size:]
print("Train:", train.index.min(), "to", train.index.max(), len(train))
print("Test:", test.index.min(), "to", test.index.max(), len(test))

## 3. Decompose trend, seasonality and residual

In [ ]:
# Seven-day period represents a weekly pattern in daily observations.
decomp = seasonal_decompose(y, model="additive", period=7, extrapolate_trend="freq")
fig = decomp.plot()
fig.set_size_inches(10, 8)
fig.suptitle("Daily Demand Decomposition: Trend / Seasonality / Residual", y=1.01)
plt.tight_layout()
plt.show()

## 4. ADF stationarity tests and differencing

In [ ]:
adf_level = adfuller(train.dropna(), autolag="AIC")
adf_first_diff = adfuller(train.diff().dropna(), autolag="AIC")
adf_results = pd.DataFrame([
    {"series":"Training level","ADF statistic":adf_level[0],"p-value":adf_level[1],"lags":adf_level[2],"nobs":adf_level[3]},
    {"series":"First difference","ADF statistic":adf_first_diff[0],"p-value":adf_first_diff[1],"lags":adf_first_diff[2],"nobs":adf_first_diff[3]}
])
display(adf_results)
print("At 5%, p > 0.05 means fail to reject the unit-root null; short samples have low test power.")

## 5. ARIMA hyperparameter selection using holdout RMSE

In [ ]:
# Small grid is intentional because the history is short.
candidates = []
for p, d, q in itertools.product(range(3), range(3), range(3)):
    order = (p, d, q)
    try:
        fitted = ARIMA(train, order=order,
                       enforce_stationarity=False,
                       enforce_invertibility=False).fit()
        pred = fitted.forecast(steps=len(test))
        rmse = np.sqrt(mean_squared_error(test, pred))
        mape = np.mean(np.abs((test - pred) / test)) * 100
        candidates.append({"order":order, "AIC":fitted.aic, "RMSE":rmse, "MAPE_%":mape})
    except Exception as exc:
        print("Skipped", order, str(exc)[:100])
scores = pd.DataFrame(candidates).sort_values(["RMSE","MAPE_%"])
display(scores)
best_order = scores.iloc[0]["order"]
best_order = tuple(int(v) for v in best_order)
print("Selected order:", best_order)

## 6. Evaluate selected ARIMA on holdout

In [ ]:
validation_model = ARIMA(train, order=best_order,
                         enforce_stationarity=False,
                         enforce_invertibility=False).fit()
holdout = validation_model.get_forecast(steps=len(test))
pred = holdout.predicted_mean
rmse = np.sqrt(mean_squared_error(test, pred))
mape = np.mean(np.abs((test - pred) / test)) * 100
mae = mean_absolute_error(test, pred)
print(f"RMSE={rmse:.3f} | MAPE={mape:.3f}% | MAE={mae:.3f}")
ci = holdout.conf_int()
plt.figure(figsize=(10,5))
plt.plot(train.index, train, label="Training")
plt.plot(test.index, test, marker="o", label="Actual holdout")
plt.plot(test.index, pred, marker="o", label=f"ARIMA{best_order}")
plt.fill_between(test.index, ci.iloc[:,0], ci.iloc[:,1], alpha=.2, label="95% interval")
plt.title("Holdout Forecast Evaluation"); plt.xlabel("Date"); plt.ylabel("Demand")
plt.legend(); plt.tight_layout(); plt.show()

## 7. Refit on full history and forecast next 30 days

In [ ]:
final_model = ARIMA(y, order=best_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False).fit()
future_result = final_model.get_forecast(steps=30)
future_mean = future_result.predicted_mean
future_ci = future_result.conf_int()
future_dates = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=30, freq="D")
future_mean.index = future_dates
future_ci.index = future_dates
forecast = pd.DataFrame({
    "date": future_dates,
    "forecast_demand": future_mean.values,
    "lower_95": future_ci.iloc[:,0].values,
    "upper_95": future_ci.iloc[:,1].values
})
display(forecast)
plt.figure(figsize=(11,5))
plt.plot(y.index, y, label="Observed")
plt.plot(future_dates, future_mean, color="tab:orange", label="30-day forecast")
plt.fill_between(future_dates, future_ci.iloc[:,0], future_ci.iloc[:,1],
                 color="tab:orange", alpha=.2, label="95% prediction interval")
plt.axvline(y.index[-1], color="gray", linestyle="--")
plt.title("30-Day Daily Demand Forecast"); plt.xlabel("Date"); plt.ylabel("Demand")
plt.legend(); plt.tight_layout(); plt.show()
forecast.to_csv("30_day_demand_forecast.csv", index=False)

## 8. Interpretation and limitations

- The ADF test is a formal stationarity check; a p-value above 0.05 means the unit-root null cannot be rejected, not that a unit root is proven.
- The model order is selected by lowest chronological holdout RMSE among the tested ARIMA candidates.
- MAPE and RMSE describe this short holdout only; six test observations are not enough for a robust estimate of generalization.
- The supplied history spans only 30 days. Weekly decomposition has roughly four cycles and the 30-day forecast is highly uncertain.
- Future marketing and holiday indicators were not used because their future values were not supplied.
- With more history, add rolling-origin validation, residual diagnostics, seasonal-naive comparison, and SARIMA/Prophet comparison.